# Olist E-commerce Data Engineering Pipeline

End-to-end batch pipeline using **BigQuery, Databricks (PySpark/Delta Lake), and Power BI**.

This notebook contains the reproducible Bronze → Silver → Gold transformations and the optional publication of Gold tables to a separate BigQuery analytics dataset. Exploratory displays and repeated validation cells from the working notebook have been removed; transformation logic is retained and grouped by layer.

> **Before running:** configure the source foreign catalog and destination dataset for your environment. Do not commit service-account JSON files, PATs, tokens, or secret values.


## 0. Imports and schemas

In [ ]:
from pyspark.sql import functions as F

# Create the medallion schemas if they do not already exist.
for layer in ["bronze", "silver", "gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS workspace.{layer}")

## 1. Bronze layer — ingest source tables

Read the raw Olist tables through the BigQuery foreign catalog and persist them as Delta tables in the Bronze schema.

In [ ]:
SOURCE_SCHEMA = "db_project_m_catalog.db_olist_dataset"

source_tables = [
    "orders",
    "customers",
    "products",
    "geolocation",
    "order_items",
    "order_payments",
    "order_reviews",
    "sellers",
]

for table_name in source_tables:
    (spark.read.table(f"{SOURCE_SCHEMA}.{table_name}")
        .write.mode("overwrite")
        .saveAsTable(f"workspace.bronze.{table_name}"))

In [ ]:
# The category translation source has generic field names in the source table.
spark.sql("""
CREATE OR REPLACE TABLE workspace.bronze.product_category_name_translation AS
SELECT
    string_field_0 AS product_category_name,
    string_field_1 AS product_category_name_english
FROM db_project_m_catalog.db_olist_dataset.p_category_name_translation
WHERE string_field_0 != 'product_category_name'
""")

## 2. Silver layer — clean and standardise

### 2.1 Customers

In [ ]:
customer_df = spark.read.table("workspace.bronze.customers")

customer_df = (
    customer_df
    .withColumn("customer_state", F.initcap(F.col("customer_state")))
    .withColumn("customer_city", F.initcap(F.col("customer_city")))
)

silver_customer_df = customer_df.dropDuplicates(["customer_id"])
(silver_customer_df.write.mode("overwrite")
    .saveAsTable("workspace.silver.silver_customer"))

### 2.2 Geolocation city standardisation

In [ ]:
city_dict = {
    # São Paulo
    "sao paulo": "Sao Paulo",
    "são paulo": "Sao Paulo",
    "saopaulo": "Sao Paulo",
    "são paulo ": "Sao Paulo",
    "sao  paulo": "Sao Paulo",

    # Rio de Janeiro
    "rio de janeiro": "Rio de Janeiro",
    "riodejaneiro": "Rio de Janeiro",
    "rio-de-janeiro": "Rio de Janeiro",
    "rio  de janeiro": "Rio de Janeiro",

    # Belo Horizonte
    "belo horizonte": "Belo Horizonte",
    "belohorizonte": "Belo Horizonte",
    "belo-horizonte": "Belo Horizonte",
    "belo  horizonte": "Belo Horizonte",

    # Curitiba
    "curitiba": "Curitiba",
    "curitba": "Curitiba",
    "curitibia": "Curitiba",
    "curitiba ": "Curitiba",

    # Porto Alegre
    "porto alegre": "Porto Alegre",
    "portoalegre": "Porto Alegre",
    "porto-alegre": "Porto Alegre",
    "porto  alegre": "Porto Alegre",

    # Salvador
    "salvador": "Salvador",
    "salvado": "Salvador",
    "salvador da bahia": "Salvador",
    "salvador-da-bahia": "Salvador",

    # Guarulhos
    "guarulhos": "Guarulhos",
    "guarulhos sp": "Guarulhos",
    "guarulhos-sp": "Guarulhos",
    "guarulhos ": "Guarulhos",

    # Brasília
    "brasilia": "Brasilia",
    "brasília": "Brasilia",
    "brasilia df": "Brasilia",
    "brasilia-df": "Brasilia",

    # São Bernardo do Campo
    "sao bernardo do campo": "Sao Bernardo do Campo",
    "são bernardo do campo": "Sao Bernardo do Campo",
    "saobernardo do campo": "Sao Bernardo do Campo",
    "sao bernardo docampo": "Sao Bernardo do Campo",
    "sao bernardo do  campo": "Sao Bernardo do Campo"
}

geo_temp_df = geo_temp_df.replace(
    city_dict,
    subset=["geolocation_city"]
)

geo_temp_df = (
    spark.read.table("workspace.bronze.geolocation")
    .withColumn("geolocation_city", F.lower(F.col("geolocation_city")))
    .replace(city_dict, subset=["geolocation_city"])
)

(geo_temp_df.write.mode("overwrite")
    .saveAsTable("workspace.silver.silver_geo"))

### 2.3 Order items

In [ ]:
o_item_temp = spark.read.table("workspace.bronze.order_items")

# Retain source rows and schema; no negative prices were removed in the working pipeline.
(o_item_temp.write.mode("overwrite")
    .saveAsTable("workspace.silver.silver_order_items"))

### 2.4 Payments

In [ ]:
payment_dict = {
    "credit_card": "Credit Card",
    "boleto": "Bank Payment",
    "voucher": "Voucher",
    "debit_card": "Debit Card",
    "not_defined": "Unknown"
}
o_pay_temp = o_pay_temp.replace(payment_dict, subset=["payment_type"])


(o_pay_temp.write.mode("overwrite")
    .saveAsTable("workspace.silver.silver_order_payments"))

### 2.5 Reviews

In [ ]:
o_reviews_temp = spark.read.table("workspace.bronze.order_reviews")

(o_reviews_temp.write.mode("overwrite")
    .saveAsTable("workspace.silver.silver_order_reviews"))

### 2.6 Orders and cancellation stage

In [ ]:
orders_temp = (
    spark.read.table("workspace.bronze.orders")
    .withColumn("order_status", F.initcap(F.col("order_status")))
)

orders_temp = orders_temp.withColumn(
    "order_cancellation_stage",
    F.when(F.col("order_status") != "Canceled", "Not Canceled")
     .when(F.col("order_approved_at").isNull(), "Before Approval")
     .when(F.col("order_delivered_carrier_date").isNull(), "After Approval")
     .when(F.col("order_delivered_customer_date").isNull(), "During Delivery")
     .otherwise("Delivered")
)

(orders_temp.write.mode("overwrite")
    .saveAsTable("workspace.silver.silver_orders"))

### 2.7 Products and English category labels

In [ ]:
p_c_name_temp = spark.read.table(
    "workspace.bronze.product_category_name_translation"
)
products_temp = spark.read.table("workspace.bronze.products")

product_f1 = (
    products_temp
    .join(p_c_name_temp, on="product_category_name", how="left")
    .drop("product_category_name")
    .withColumnRenamed("product_category_name_english", "product_category_name")
)

for column in [
    "product_name_lenght", "product_description_lenght", "product_photos_qty",
    "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"
]:
    product_f1 = product_f1.withColumn(column, F.col(column).cast("double"))

product_f1 = (
    product_f1
    .fillna(0, subset=["product_description_lenght", "product_name_lenght", "product_photos_qty"])
    .fillna("No Category", subset=["product_category_name"])
    .replace({"Baby": "Baby Care"}, subset=["product_category_name"])
    .withColumn("product_category_name", F.regexp_replace("product_category_name", "_", " "))
    .withColumn("product_category_name", F.initcap(F.col("product_category_name")))
)

(product_f1.write.mode("overwrite")
    .saveAsTable("workspace.silver.silver_product"))

### 2.8 Sellers and city standardisation

In [ ]:
city_dict_2 = {
    "sao paulo sp": "Sao Paulo",
    "sp / sp": "Sao Paulo",
    "sao pauo": "Sao Paulo",
    "sao paulo": "Sao Paulo",
    "sao paulo / sao paulo": "Sao Paulo",
    "sao paulop" : "Sao Paulo",
    "sao paulo - sp" : "Sao Paulo",
    "sp" : "Sao Paulo",
    "sao  paulo" : "Sao Paulo",
    "são paulo" : "Sao Paulo",
    "sao paluo" : "Sao Paulo",
    "rio de janeiro / rio de janeiro" : "Rio de Janeiro",
    "rio de janeiro" : "Rio de Janeiro",
    "rio de janeiro - rj" : "Rio de Janeiro",
    "rj" : "Rio de Janeiro",
    "rio de janeiro /rio de janeiro" :"Rio de Janeiro",
    "sao jose dos pinhais":"Sao Jose dos Pinhais",
    "sao jose dos pinhas":"Sao Jose dos Pinhais",
    "sao  jose dos pinhais" : "Sao Jose dos Pinhais",
    "balenario camboriu": "Balneario Camboriu",
    "balneario camboriu" : "Balneario Camboriu"
}



seller_temp = spark.read.table("workspace.bronze.sellers")
seller_temp = (
    seller_temp
    .replace(city_dict_2, subset=["seller_city"])
    .withColumn("seller_city", F.initcap(F.col("seller_city")))
    .withColumn(
        "seller_city",
        F.when(F.col("seller_city").rlike(r"[/\\]"), "Multiple Cities")
         .otherwise(F.col("seller_city"))
    )
)

(seller_temp.write.mode("overwrite")
    .saveAsTable("workspace.silver.silver_sellers"))

## 3. Gold layer — business-level analytical tables

### 3.1 Sales — one row per order item

In [ ]:
s_orders = spark.read.table("workspace.silver.silver_orders")
s_product = spark.read.table("workspace.silver.silver_product")
s_seller = spark.read.table("workspace.silver.silver_sellers")
s_oi = spark.read.table("workspace.silver.silver_order_items")

gold_sales = (
    s_oi
    .join(s_orders, on="order_id", how="left")
    .join(s_product, on="product_id", how="left")
    .join(s_seller, on="seller_id", how="left")
    .select(
        "order_id", "customer_id", "order_status", "order_purchase_timestamp",
        "order_cancellation_stage", "product_category_name", "order_item_id",
        "product_id", "seller_id", "price", "freight_value", "seller_city", "seller_state"
    )
    .withColumn("item_total_value", F.round(F.col("price") + F.col("freight_value"), 2))
    .withColumn("order_purchase_date", F.to_date(F.col("order_purchase_timestamp")))
    .withColumn("order_purchase_month", F.month(F.col("order_purchase_date")))
)

# Add the stable customer-level identifier used for customer analysis.
cust_temp = spark.read.table("workspace.silver.silver_customer")
gold_sales = gold_sales.join(
    cust_temp.select("customer_id", "customer_unique_id"),
    on="customer_id", how="left"
)

(gold_sales.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.gold_sales"))

### 3.2 Customers — one row per customer_unique_id

In [ ]:
customer_metrics = (
    gold_sales
    .groupBy("customer_unique_id")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.count("order_item_id").alias("total_items"),
        F.round(F.sum("price"), 2).alias("total_spend"),
        F.round(F.sum("freight_value"), 2).alias("total_freight"),
        F.round(F.sum("item_total_value"), 2).alias("total_order_value"),
        F.countDistinct(F.when(F.col("order_status") == "Canceled", F.col("order_id"))).alias("canceled_orders"),
        F.min("order_purchase_date").alias("first_order_date"),
        F.max("order_purchase_date").alias("last_order_date")
    )
    .withColumn(
        "Avg_order_value",
        F.round(F.col("total_order_value") / F.col("total_orders"), 2)
    )
)

customer_info = (
    cust_temp.groupBy("customer_unique_id")
    .agg(
        F.first("customer_city", ignorenulls=True).alias("customer_city"),
        F.first("customer_state", ignorenulls=True).alias("customer_state"),
        F.first("customer_zip_code_prefix", ignorenulls=True).alias("customer_zip_code_prefix")
    )
)

gold_customers = customer_info.join(customer_metrics, on="customer_unique_id", how="left")
(gold_customers.write.mode("overwrite")
    .saveAsTable("workspace.gold.gold_customers"))

### 3.3 Products — one row per product_id

The average review score is derived from reviews on orders containing the product; it is not a product-specific review entity in the source data.

In [ ]:
silver_o_items = spark.read.table("workspace.silver.silver_order_items")
silver_products = spark.read.table("workspace.silver.silver_product")
silver_reviews = spark.read.table("workspace.silver.silver_order_reviews")

silver_reviews_temp = (
    silver_reviews.groupBy("order_id")
    .agg(F.avg("review_score").alias("avg_review_score"))
)

order_items_review = silver_o_items.join(silver_reviews_temp, on="order_id")
product_orderitem_review_temp = silver_products.join(order_items_review, on="product_id")

gold_product = (
    product_orderitem_review_temp
    .groupBy("product_id", "product_category_name")
    .agg(
        F.count("order_item_id").alias("total_units_sold"),
        F.countDistinct("order_id").alias("total_orders"),
        F.round(F.sum("price"), 2).alias("total_revenue"),
        F.round(F.avg("price"), 2).alias("avg_product_price"),
        F.round(F.avg("freight_value"), 2).alias("avg_freight_value"),
        F.round(F.avg("avg_review_score"), 2).alias("avg_review_score")
    )
)

(gold_product.write.mode("overwrite")
    .saveAsTable("workspace.gold.gold_product"))

### 3.4 Delivery — one row per order

In [ ]:
silver_order_temp = spark.read.table("workspace.silver.silver_orders")

gold_delivery = (
    silver_order_temp
    .withColumn(
        "delivery_days",
        F.when(
            F.col("order_delivered_customer_date").isNotNull(),
            F.datediff(F.to_date("order_delivered_customer_date"), F.to_date("order_purchase_timestamp"))
        )
    )
    .withColumn(
        "estimated_delivery_days",
        F.datediff(F.to_date("order_estimated_delivery_date"), F.to_date("order_purchase_timestamp"))
    )
    .withColumn(
        "delivery_delay_days",
        F.when(
            F.col("order_delivered_customer_date").isNotNull() & F.col("order_estimated_delivery_date").isNotNull(),
            F.datediff(F.to_date("order_delivered_customer_date"), F.to_date("order_estimated_delivery_date"))
        )
    )
)

# Add customer state for geographic delivery analysis.
customer_states = (
    cust_temp.select("customer_id", "customer_state")
    .dropDuplicates(["customer_id"])
)
gold_delivery = gold_delivery.join(customer_states, on="customer_id", how="left")

(gold_delivery.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.gold_delivery"))

### 3.5 Payment methods — one row per order_id + payment_type

In [ ]:
s_o_pay = spark.read.table("workspace.silver.silver_order_payments")

gold_payment_methods = (
    s_o_pay.groupBy("order_id", "payment_type")
    .agg(
        F.round(F.sum("payment_value"), 2).alias("total_payment_value"),
        F.count("*").alias("payment_records"),
        F.max("payment_installments").alias("max_installments")
    )
)

(gold_payment_methods.write.format("delta").mode("overwrite")
    .saveAsTable("workspace.gold.gold_payment_methods"))

### 3.6 Order payments — one row per order_id

In [ ]:
gold_order_payments = (
    s_o_pay.groupBy("order_id")
    .agg(
        F.round(F.sum("payment_value"), 2).alias("total_payment_value"),
        F.count("*").alias("total_payment_records"),
        F.max("payment_installments").alias("max_installments"),
        F.countDistinct("payment_type").alias("payment_method_count")
    )
)

(gold_order_payments.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.gold_order_payments"))

## 4. Gold data-quality checks

In [ ]:
# Expected grain / uniqueness checks for all six Gold tables.
key_checks = {
    "gold_sales": ["order_id", "order_item_id"],
    "gold_customers": ["customer_unique_id"],
    "gold_product": ["product_id"],
    "gold_delivery": ["order_id"],
    "gold_order_payments": ["order_id"],
    "gold_payment_methods": ["order_id", "payment_type"]
}

for table_name, key_columns in key_checks.items():
    df = spark.table(f"workspace.gold.{table_name}")
    total_rows = df.count()
    unique_keys = df.select(*key_columns).distinct().count()
    status = "PASS" if total_rows == unique_keys else "FAIL"
    print(f"{table_name}: {status} | rows={total_rows} | unique keys={unique_keys}")

## 5. Publish Gold tables to BigQuery (optional)

This project publishes only the Gold layer to a separate analytics dataset. The approach below uses the BigQuery Python client and Pandas. It collects each Spark DataFrame to the driver, so use it for suitably sized tables; for larger datasets, prefer a distributed Spark connector or staged export.

In [ ]:
%pip install google-cloud-bigquery

In [ ]:
import json
from google.cloud import bigquery
from google.oauth2 import service_account

# Retrieve the service-account JSON from Databricks Secrets. Never print this value.
service_account_info = json.loads(
    dbutils.secrets.get(scope="gcp-credentials", key="gcp-service-account-json")
)
credentials = service_account.Credentials.from_service_account_info(
    service_account_info,
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
client = bigquery.Client(
    project="databricksproject-510805",
    credentials=credentials
)

# Test destination dataset access.
dataset = client.get_dataset("databricksproject-510805.db_olist_analytics")
print("BigQuery connection successful:", dataset.dataset_id)

In [ ]:
DESTINATION_DATASET = "databricksproject-510805.db_olist_analytics"

def publish_gold_table(source_table):
    spark_df = spark.table(f"workspace.gold.{source_table}")
    pandas_df = spark_df.toPandas()
    destination_table = f"{DESTINATION_DATASET}.{source_table}"

    job_config = bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
    )
    job = client.load_table_from_dataframe(
        pandas_df, destination_table, job_config=job_config
    )
    job.result()
    print(f"Published {source_table}: {len(pandas_df):,} rows")

for table_name in [
    "gold_sales", "gold_customers", "gold_product",
    "gold_delivery", "gold_order_payments", "gold_payment_methods"
]:
    publish_gold_table(table_name)

## 6. BigQuery publication validation

In [ ]:
# Compare destination row counts with the source Gold tables.
for table_name in [
    "gold_sales", "gold_customers", "gold_product",
    "gold_delivery", "gold_order_payments", "gold_payment_methods"
]:
    source_count = spark.table(f"workspace.gold.{table_name}").count()
    destination = client.get_table(f"{DESTINATION_DATASET}.{table_name}")
    status = "PASS" if source_count == destination.num_rows else "FAIL"
    print(
        f"{table_name}: {status} | Databricks={source_count:,} "
        f"| BigQuery={destination.num_rows:,}"
    )

## Notes and interpretation

- `gold_sales` grain: one row per order item (`order_id`, `order_item_id`).
- `gold_customers` grain: one row per `customer_unique_id`.
- `gold_product` grain: one row per `product_id`. Its `avg_review_score` is based on order reviews joined to items in orders; it should not be described as a direct product review score.
- `gold_delivery` grain: one row per order. `delivery_delay_days` is delivered date minus estimated date: positive means late, negative means early.
- `gold_order_payments` grain: one row per order with payment records; it can have fewer rows than the delivery table when an order has no payment record.
- `gold_payment_methods` grain: one row per order and payment type.
- `WRITE_TRUNCATE` replaces the destination table on each publication run.
- This notebook assumes the foreign catalog, source dataset, secret scope, secret key, destination dataset, and required cloud permissions have already been configured.
